# Faithfulness pilot (Colab)
Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom.

In [ ]:
# 1. Get the code (if the repo is private, use https://<TOKEN>@github.com/Aman14o5/BTech-Project.git)
!git clone -b faithfulness-pilot https://github.com/Aman14o5/BTech-Project.git
%cd BTech-Project
!pip -q install transformers sentencepiece accelerate

# 2. Get a fine-tuned T5-Large e-SNLI checkpoint
Open the authors' Google Drive folder (link in the repo README) → find a T5-Large model fine-tuned on **e-SNLI** (e.g. `random / sub0 / nt8` or similar) →
right-click → *Organize → Add shortcut to My Drive*. Then mount Drive and set `MODEL_PATH` to that folder.
The folder must contain `config.json` and the model weights (`pytorch_model.bin` or `model.safetensors`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = '/content/drive/MyDrive/<path-to>/esnli/random/sub0/nt8'   # <-- edit
!ls "$MODEL_PATH" 

In [ ]:
# 3. Sanity check: one prediction
import sys; sys.path.insert(0, 'faithfulness')
from predictors import T5Predictor
pred = T5Predictor(MODEL_PATH)
pred.predict([("A man getting a tattoo on his back.", "A woman is getting a tattoo.")])

In [ ]:
# 4. Pilot run (300 examples ≈ a few minutes on a T4)
!python faithfulness/run_pilot.py --model_path "$MODEL_PATH" --n 300 --out_dir faithfulness/results/pilot_t5_nt8

In [ ]:
# 5. Look at examples flagged as possibly unfaithful
import pandas as pd
df = pd.read_csv('faithfulness/results/pilot_t5_nt8/per_example.csv')
pd.set_option('display.max_colwidth', 120)
df[df.suspect_unfaithful].head(15)

In [ ]:
# 6. Save results back to Drive (Colab disk is wiped when the session ends)
!cp -r faithfulness/results /content/drive/MyDrive/btp_results